# Файнтюн чемпиона Qwen3-32B QLoRA в Google Colab

Прогон **одной** лучшей модели по итогам 3 раундов на GPU-сервере с двумя свежими исправлениями:
1. **Class weights в loss** (`use_class_weights=true`) — главный путь к +3-7pt на macro_f1
2. **Train/val split 90/10 stratified** (`val_split=0.10`) — устраняет selection bias

**Что нужно:**
- Colab Pro / Pro+ с **A100-40GB** или **L4-24GB** (профиль L4 уже подготовлен)
- Папка на Google Drive `MyDrive/VKR/Data/` с CSV-данными

**Что сохранится на Drive:** адаптер, results CSV, manifest.json. Логи Trainer печатаются прямо в output ячейки.

## 1. Монтируем Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = Path('/content/drive/MyDrive/VKR')
DRIVE_DATA = DRIVE_ROOT / 'Data'
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_RUN = DRIVE_ROOT / 'finetune_runs' / RUN_TS
DRIVE_RUN.mkdir(parents=True, exist_ok=True)

print(f'Drive root      : {DRIVE_ROOT} (exists={DRIVE_ROOT.exists()})')
print(f'Drive Data      : {DRIVE_DATA} (exists={DRIVE_DATA.exists()})')
print(f'Run output      : {DRIVE_RUN}')

assert DRIVE_DATA.exists(), (
    f'Папка с данными {DRIVE_DATA} не найдена. '
    'Положи на Drive: data_after_stage3.csv, data_test.csv, train_after_eda.csv.'
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root      : /content/drive/MyDrive/VKR (exists=True)
Drive Data      : /content/drive/MyDrive/VKR/Data (exists=True)
Run output      : /content/drive/MyDrive/VKR/finetune_runs/20260430_123829


## 2. Клонируем / обновляем код из GitHub

In [2]:
# === Если репо ПРИВАТНЫЙ — раскомментируй и подставь PAT ===
# GITHUB_PAT = 'ghp_xxxxxxxxxxxxxxxxxxxx'
# REPO_URL = f'https://{GITHUB_PAT}@github.com/KVTur23/llm-finetuning-text-classification.git'

# === Публичный репо ===
REPO_URL = 'https://github.com/KVTur23/llm-finetuning-text-classification.git'
REPO_BRANCH = 're-augmentation'
REPO_DIR = Path('/content/VKR')

In [3]:
import subprocess

if (REPO_DIR / '.git').exists():
    print('Репо уже клонирован — git pull')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', REPO_BRANCH], check=True)
else:
    print(f'Клонирую {REPO_URL} в {REPO_DIR}')
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

commit = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'],
    capture_output=True, text=True, check=True,
).stdout.strip()
print(f'\nТекущий коммит: {commit}')

Репо уже клонирован — git pull

Текущий коммит: 8a75953 data


## 3. Подцепляем данные с Drive (symlink)

In [4]:
PROJECT_CODE = REPO_DIR / 'code'
LOCAL_DATA = PROJECT_CODE / 'Data'

# Если в репо есть пустая Data/ — переименовываем, чтобы не конфликтовала
if LOCAL_DATA.exists() and not LOCAL_DATA.is_symlink():
    backup = LOCAL_DATA.parent / 'Data_repo_orig'
    if not backup.exists():
        LOCAL_DATA.rename(backup)
    else:
        import shutil; shutil.rmtree(LOCAL_DATA)

LOCAL_DATA.mkdir(exist_ok=True)
for csv in DRIVE_DATA.glob('*.csv'):
    target = LOCAL_DATA / csv.name
    if not target.exists():
        target.symlink_to(csv)

(LOCAL_DATA / 'finetune_checkpoints').mkdir(exist_ok=True)
(PROJECT_CODE / 'results').mkdir(exist_ok=True)

print('Data в репо:')
for f in sorted(LOCAL_DATA.iterdir()):
    if f.is_symlink():
        print(f'  {f.name} -> {f.readlink()}')
    elif f.is_dir():
        print(f'  {f.name}/ ({len(list(f.iterdir()))} items)')

Data в репо:
  _stage3_pairs_cache.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache.bak.csv
  _stage3_pairs_cache_deu_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_deu_Latn.bak.csv
  _stage3_pairs_cache_eng_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_eng_Latn.bak.csv
  _stage3_pairs_cache_fra_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_fra_Latn.bak.csv
  _stage3_pairs_cache_round1_deu_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_round1_deu_Latn.bak.csv
  _stage3_pairs_cache_round1_eng_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_round1_eng_Latn.bak.csv
  _stage3_pairs_cache_round1_fra_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_round1_fra_Latn.bak.csv
  _stage3_pairs_cache_round2_deu_Latn.bak.csv -> /content/drive/MyDrive/VKR/Data/_stage3_pairs_cache_round2_deu_Latn.bak.csv
  _stage3_pairs_cache_round2_eng_Latn.bak.csv -> /content/dri

## 4. Устанавливаем зависимости

In [5]:
!pip install -q transformers==4.57.6 peft bitsandbytes accelerate datasets sentencepiece scikit-learn pandas filelock


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 115.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.7 MB/s eta 0:00:00


## 5. Проверка GPU + sys.path

In [5]:
import torch
import sys

if str(PROJECT_CODE) not in sys.path:
    sys.path.insert(0, str(PROJECT_CODE))

if not torch.cuda.is_available():
    raise RuntimeError('CUDA недоступна. Runtime → Change runtime type → A100/L4 GPU')

name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  ({vram_gb:.1f} GB)')

if vram_gb >= 75:
    GPU_PROFILE = 'A100_80'
elif vram_gb >= 38:
    GPU_PROFILE = 'A100_40'
elif vram_gb >= 22:
    GPU_PROFILE = 'L4'
    print('⚠ L4 24GB — впритык, но профиль уже подготовлен (batch=1, max_seq=1536).')
else:
    raise RuntimeError(f'GPU слишком маленькая ({vram_gb:.1f} GB). Нужно ≥24GB.')

print(f'Профиль: {GPU_PROFILE}')

GPU: NVIDIA A100-SXM4-40GB  (42.4 GB)
Профиль: A100_40


## 6. Финализируем конфиг — class_weights + val_split

In [ ]:
import json, shutil

# Все эксперименты теперь самодостаточные конфиги в config_models/finetune_configs/.
# Тут просто выбираем нужный и показываем что внутри - никаких overrides.
config_path = PROJECT_CODE / 'config_models' / 'finetune_configs' / 'qwen3_32b_qlora_cw.json'

with open(config_path) as f:
    cfg = json.load(f)

shutil.copy(config_path, DRIVE_RUN / 'config_used.json')

print('=== config ===')
print(json.dumps(cfg, ensure_ascii=False, indent=2))

## 7. Sanity-check + cd

In [ ]:
%cd {PROJECT_CODE}

# Проверяем что нужный конфиг есть на диске
config_path = PROJECT_CODE / 'config_models' / 'finetune_configs' / 'qwen3_32b_qlora_cw.json'
assert config_path.exists(), f'Не найден конфиг: {config_path}'
print(f'Конфиг: {config_path.name}')

# Если уже был прогон с тем же run_key - результат лежит в results/finetune/
results_json = PROJECT_CODE / 'results' / 'finetune' / 'qwen3_32b_qlora_cw.json'
if results_json.exists():
    import json
    with open(results_json) as f:
        prev = json.load(f)
    print(f'\nУже был прогон: bal_acc={prev["balanced_accuracy"]:.4f}, macro_f1={prev["macro_f1"]:.4f}')
    print('Запуск ниже перезапишет файл.')
else:
    print('\nПрогон чистый - results/finetune/qwen3_32b_qlora_cw.json создастся в конце.')

## 8. Прогон тренировки

**Время**: A100-40 ~3 ч, L4 ~5 ч. Логи Trainer (loss / eval_balanced_accuracy / eval_macro_f1 по эпохам) пойдут прямо в output ячейки — можно следить.

In [ ]:
!python scripts/run_finetune.py --config qwen3_32b_qlora_cw.json --gpu {GPU_PROFILE}

## 9. Сохраняем адаптер + results на Drive

In [ ]:
import shutil

RUN_KEY = 'qwen3_32b_qlora_cw'

# 1. Адаптер
src_adapter = PROJECT_CODE / 'Data' / 'finetune_checkpoints' / RUN_KEY
dst_adapter = DRIVE_RUN / 'adapter'
if src_adapter.exists():
    print(f'Копирую адаптер -> Drive (1-3 минуты)...')
    shutil.copytree(src_adapter, dst_adapter, dirs_exist_ok=True)
    sz = sum(f.stat().st_size for f in dst_adapter.rglob('*') if f.is_file()) / 1e6
    print(f'OK Адаптер: {sz:.1f} МБ')
else:
    print(f'! Адаптер не найден: {src_adapter}')

# 2. Test curve CSV (per-epoch test metrics)
test_curve = src_adapter / 'test_curve.csv' if src_adapter.exists() else None
if test_curve and test_curve.exists():
    shutil.copy(test_curve, dst_adapter / 'test_curve.csv')
    print(f'OK Test curve -> {dst_adapter / "test_curve.csv"}')

# 3. Results: per-run JSON + per-sample preds CSV
src_results = PROJECT_CODE / 'results'
dst_results = DRIVE_RUN / 'results'
shutil.copytree(src_results, dst_results, dirs_exist_ok=True)
print(f'OK Results -> {dst_results}')

# 4. Manifest для воспроизводимости
result_json = PROJECT_CODE / 'results' / 'finetune' / f'{RUN_KEY}.json'
metrics = json.loads(result_json.read_text()) if result_json.exists() else {}

manifest = {
    'timestamp': RUN_TS,
    'gpu': name,
    'gpu_profile': GPU_PROFILE,
    'commit': commit,
    'run_key': RUN_KEY,
    'config': cfg,
    'metrics': metrics,
}
with open(DRIVE_RUN / 'manifest.json', 'w') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f'OK Manifest -> {DRIVE_RUN / "manifest.json"}')

## 10. Финальные метрики + сравнение с baseline R1

In [ ]:
import json

RUN_KEY = 'qwen3_32b_qlora_cw'
result_json = PROJECT_CODE / 'results' / 'finetune' / f'{RUN_KEY}.json'

if result_json.exists():
    with open(result_json) as f:
        r = json.load(f)
    print(f'=== {RUN_KEY} ===')
    for k in ('balanced_accuracy', 'macro_f1', 'f1_group_A', 'f1_group_B', 'f1_group_C'):
        print(f'  {k:20s} = {r[k]:.4f}')
    print(f'  trainable_params     = {r["trainable_params"]}')
    print(f'  train_time_sec       = {r["train_time_sec"]:.1f}')
else:
    print(f'! {result_json} не найден - тренировка не завершилась успешно')

---
## Diagnostic — если ячейка 8 завершилась слишком быстро (<1 мин)

Запусти эту ячейку чтобы посмотреть, что не так:

In [ ]:
import json

# Если ячейка тренировки упала - смотрите её stderr выше.
# Здесь просто показываем что лежит в results/finetune/ после прогона.
results_dir = PROJECT_CODE / 'results' / 'finetune'
if results_dir.exists():
    files = sorted(results_dir.glob('*.json'))
    print(f'results/finetune/: {len(files)} файлов')
    for f in files:
        with open(f) as fh:
            r = json.load(fh)
        print(f'  {f.name}: bal_acc={r.get("balanced_accuracy", "-"):.4f}, macro_f1={r.get("macro_f1", "-"):.4f}')
else:
    print('results/finetune/ пока не существует - тренировка не завершилась успешно.')

## 11. Eval восстановленных адаптеров (recovery после краша 2026-04-29)

Краш на сохранении `checkpoint-396` забил оба диска (Colab + Drive). Адаптеры спасены вручную:
`MyDrive/VKR/recovered_weights/qlora_qwen3_32b_e{2,3}/`.

- **E2** (epoch 2, был best by val macro_f1 = 0.787)
- **E3** (epoch 3, val метрик нет — краш до eval-фазы)

Ниже — гонка обоих адаптеров на тесте через `evaluate_finetuned.evaluate()`.


In [ ]:
# Sanity-check восстановленных адаптеров на Drive
RECOVERED_DIR = DRIVE_ROOT / 'recovered_weights'
ADAPTER_E2 = RECOVERED_DIR / 'qlora_qwen3_32b_e2'
ADAPTER_E3 = RECOVERED_DIR / 'qlora_qwen3_32b_e3'

assert ADAPTER_E2.exists(), f'Не найдено: {ADAPTER_E2}'
assert ADAPTER_E3.exists(), f'Не найдено: {ADAPTER_E3}'

REQUIRED = [
    'adapter_model.safetensors', 'adapter_config.json',
    'id2label.json', 'class_groups.json',
    'tokenizer.json', 'tokenizer_config.json',
]
for d in (ADAPTER_E2, ADAPTER_E3):
    missing = [f for f in REQUIRED if not (d / f).exists()]
    assert not missing, f'{d.name}: не хватает {missing}'
    sz = (d / 'adapter_model.safetensors').stat().st_size / 1e9
    print(f'  {d.name}: adapter={sz:.2f} GB ✓')


In [ ]:
# Eval E2 + E3 → results/finetune_results.csv + per-sample preds
import gc, json, os, shutil
import torch
import pandas as pd

# CWD = PROJECT_CODE — иначе evaluate пишет в неправильное место (relative paths)
os.chdir(PROJECT_CODE)

with open(PROJECT_CODE / 'config_models' / 'pipeline_config.json') as f:
    pipeline_cfg = json.load(f)
pipeline_cfg['gpu_name'] = GPU_PROFILE

CONFIG_PATH = PROJECT_CODE / 'config_models' / 'finetune_configs' / 'qwen3_32b_qlora.json'

from src.finetune.evaluate_finetuned import evaluate

results = {}
for tag, adapter_dir in [('e2', ADAPTER_E2), ('e3', ADAPTER_E3)]:
    run_key = f'qlora_qwen3_32b_recovered_{tag}'
    print(f'\n{"="*60}\nEvaluate {tag.upper()}: {adapter_dir.name}\n{"="*60}')
    res = evaluate(
        adapter_dir=str(adapter_dir),
        config_path=str(CONFIG_PATH),
        pipeline_cfg=pipeline_cfg,
        run_key=run_key,
    )
    results[tag] = res

    # Освобождаем VRAM перед загрузкой следующего base+adapter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _fmt(v):
    return round(v, 4) if v is not None else None

summary = pd.DataFrame([
    {
        'epoch': k.upper(),
        'bal_acc':  _fmt(v['balanced_accuracy']),
        'macro_f1': _fmt(v['macro_f1']),
        'f1_A': _fmt(v['f1_group_A']),
        'f1_B': _fmt(v['f1_group_B']),
        'f1_C': _fmt(v['f1_group_C']),
    }
    for k, v in results.items()
])
print('\n=== Сводка восстановленных метрик ===')
print(summary.to_string(index=False))

# Backup на Drive — чтобы не потерять снова
RECOVERED_OUT = DRIVE_ROOT / 'recovered_results'
RECOVERED_OUT.mkdir(exist_ok=True)
summary.to_csv(RECOVERED_OUT / 'qlora_qwen3_32b_recovered_summary.csv', index=False)
for tag in ('e2', 'e3'):
    p = PROJECT_CODE / 'results' / f'preds_qlora_qwen3_32b_recovered_{tag}.csv'
    if p.exists():
        shutil.copy(p, RECOVERED_OUT / p.name)
results_csv = PROJECT_CODE / 'results' / 'finetune_results.csv'
if results_csv.exists():
    shutil.copy(results_csv, RECOVERED_OUT / 'finetune_results.csv')
print(f'\nBackup → {RECOVERED_OUT}')
